# Week 04 — FastAPI Backend Fundamentals
## LeafGuard AI: Building the Disease Detection API

This notebook walks you through building the FastAPI backend for LeafGuard AI.
Run each cell, read the explanations, and experiment with the code.

**Prerequisites:** Python 3.8+, pip installed

**Learning outcomes:**
- Understand what a REST API is and how HTTP works
- Build a FastAPI application from scratch
- Handle image file uploads
- Write Pydantic models for request/response validation
- Test endpoints with Python requests library

---

## Cell 1: Install Dependencies

Run this cell once to install all required packages.
In a real project these are listed in `requirements.txt`.

In [ ]:
# Install FastAPI and its dependencies
# Uncomment and run if not already installed
# !pip install fastapi uvicorn python-multipart Pillow pydantic requests

# Verify installations
import fastapi
import pydantic
import PIL
print(f'FastAPI version: {fastapi.__version__}')
print(f'Pydantic version: {pydantic.__version__}')
print(f'PIL (Pillow) version: {PIL.__version__}')
print('All dependencies available!')

## Cell 2: HTTP Basics — Understanding the Protocol

Before building an API, understand the underlying HTTP protocol.

HTTP is a request-response protocol:
- **Client** (Android app) sends a **Request**: method + URL + headers + optional body
- **Server** (FastAPI) sends a **Response**: status code + headers + body

```
Android App                        FastAPI Server
    |                                    |
    |  POST /predict HTTP/1.1           |
    |  Host: 192.168.1.105:8000         |
    |  Content-Type: multipart/form-data|
    |  [image bytes]                    |
    | --------------------------------> |
    |                                    |
    |  HTTP/1.1 200 OK                  |
    |  Content-Type: application/json   |
    |  {"disease": "Early Blight", ...} |
    | <-------------------------------- |
```

In [ ]:
# Simulate an HTTP request/response cycle using Python's requests library
# This is what Retrofit does under the hood on Android

import json

# Simulate a request object (what the server receives)
simulated_request = {
    'method': 'POST',
    'url': 'http://localhost:8000/predict',
    'headers': {
        'Content-Type': 'multipart/form-data; boundary=----boundary123',
        'User-Agent': 'LeafGuard-Android/1.0',
        'Accept': 'application/json'
    },
    'body': '[binary image data]'
}

# Simulate a response object (what Android receives)
simulated_response = {
    'status_code': 200,
    'headers': {'Content-Type': 'application/json'},
    'body': {
        'success': True,
        'prediction': {
            'disease': 'Tomato Early Blight',
            'confidence': 0.91,
            'is_healthy': False
        }
    }
}

print('REQUEST sent by Android:')
print(f"  {simulated_request['method']} {simulated_request['url']}")
for key, val in simulated_request['headers'].items():
    print(f'  {key}: {val}')

print()
print('RESPONSE received by Android:')
print(f"  Status: {simulated_response['status_code']} OK")
print(f"  Body: {json.dumps(simulated_response['body'], indent=2)}")

## Cell 3: Your First FastAPI Application

Let's build a minimal FastAPI app and understand each line.

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

# Create the FastAPI application instance
# title and description appear in the auto-generated /docs page
app = FastAPI(
    title='LeafGuard AI API',
    description='Plant disease detection via ML model inference',
    version='1.0.0'
)

# Configure CORS — allows browser-based clients to call this API
# Android apps ignore CORS, but browsers enforce it
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],      # All origins (development only)
    allow_methods=['*'],
    allow_headers=['*'],
)

# Health check endpoint — used by Android to verify server is running
@app.get('/health')
async def health_check():
    return {
        'status': 'healthy',
        'version': '1.0.0',
        'model_loaded': False  # Will be True once model is integrated
    }

# Root endpoint
@app.get('/')
async def root():
    return {'message': 'LeafGuard AI API is running'}

print('FastAPI app created successfully!')
print(f'App title: {app.title}')
print(f'App version: {app.version}')
print()
print('Routes registered:')
for route in app.routes:
    if hasattr(route, 'methods'):
        print(f'  {list(route.methods)} {route.path}')

## Cell 4: Pydantic Models — Type-safe Request/Response

Pydantic models define the exact shape of JSON data your API accepts and returns.
FastAPI uses them to validate automatically and generate API documentation.

In [ ]:
from pydantic import BaseModel, Field, validator
from typing import List, Dict, Optional

# Response model for a single disease detection
class DiseaseResult(BaseModel):
    disease: str = Field(..., description='Full disease name, e.g. Tomato Early Blight')
    confidence: float = Field(..., ge=0.0, le=1.0, description='Confidence 0.0 to 1.0')
    is_healthy: bool
    symptoms: List[str] = []
    treatment: str = ''
    prevention: List[str] = []

# Full prediction response
class PredictionResponse(BaseModel):
    success: bool
    prediction: DiseaseResult
    processing_time_ms: float
    model_version: str = '1.0'

# Error response
class ErrorResponse(BaseModel):
    success: bool = False
    error_code: str
    message: str

# Create a sample response and validate it
sample_response = PredictionResponse(
    success=True,
    prediction=DiseaseResult(
        disease='Tomato Early Blight',
        confidence=0.91,
        is_healthy=False,
        symptoms=['Brown spots with concentric rings', 'Yellow halos', 'Leaf drop'],
        treatment='Prune infected leaves. Apply copper-based fungicide.',
        prevention=['Avoid overhead watering', 'Rotate crops seasonally']
    ),
    processing_time_ms=42.7
)

# Pydantic provides .dict() and .json() serialisation
import json
print('Validated response:')
print(json.dumps(sample_response.dict(), indent=2))

# Test validation failure
print()
print('Testing validation (confidence = 1.5 should fail):')
try:
    invalid = DiseaseResult(disease='Test', confidence=1.5, is_healthy=False)
except Exception as e:
    print(f'Validation error caught: {type(e).__name__}')
    print('FastAPI returns this as HTTP 422 automatically!')

## Cell 5: File Upload Endpoint

The core endpoint that Android calls to detect disease from a photo.

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from PIL import Image
import io
import time

# Allowed image MIME types
ALLOWED_TYPES = {'image/jpeg', 'image/png', 'image/webp'}
MAX_FILE_SIZE = 10 * 1024 * 1024  # 10 MB

# Mock disease database (will be replaced by real ML model)
MOCK_DISEASES = [
    {'name': 'Tomato Early Blight', 'confidence': 0.91, 'healthy': False},
    {'name': 'Tomato Healthy', 'confidence': 0.95, 'healthy': True},
    {'name': 'Corn Common Rust', 'confidence': 0.88, 'healthy': False},
]

import random

# This function simulates what main.py does
async def handle_prediction_upload(file: UploadFile):
    start_time = time.time()

    # Step 1: Validate content type
    if file.content_type not in ALLOWED_TYPES:
        raise HTTPException(
            status_code=400,
            detail=f'Invalid file type: {file.content_type}. Use JPEG or PNG.'
        )

    # Step 2: Read bytes
    contents = await file.read()

    # Step 3: Validate size
    if len(contents) > MAX_FILE_SIZE:
        raise HTTPException(status_code=413, detail='File too large. Maximum 10 MB.')

    # Step 4: Decode image
    try:
        image = Image.open(io.BytesIO(contents)).convert('RGB')
    except Exception:
        raise HTTPException(status_code=400, detail='Cannot decode image file.')

    # Step 5: Simulate model inference (random for demo)
    mock_result = random.choice(MOCK_DISEASES)

    processing_ms = (time.time() - start_time) * 1000

    return {
        'success': True,
        'image_size': f'{image.width}×{image.height}',
        'file_size_kb': len(contents) / 1024,
        'prediction': mock_result,
        'processing_time_ms': round(processing_ms, 1)
    }

# Demonstrate image processing logic
from PIL import Image as PILImage
import numpy as np

# Create a synthetic test image (224x224 green leaf-like)
test_img = PILImage.new('RGB', (224, 224), color=(34, 139, 34))

# Add some texture (dots to simulate a leaf with spots)
pixels = np.array(test_img)
for _ in range(20):
    x, y = random.randint(0, 223), random.randint(0, 223)
    size = random.randint(5, 15)
    pixels[max(0,y-size):y+size, max(0,x-size):x+size] = [101, 67, 33]  # Brown spots

test_img = PILImage.fromarray(pixels.astype(np.uint8))

# Simulate preprocessing
img_array = np.array(test_img, dtype=np.float32) / 255.0  # Normalise
img_batch = np.expand_dims(img_array, axis=0)              # Add batch dim

print(f'Original image size: {test_img.size}')
print(f'Image array shape: {img_array.shape}')
print(f'Batch tensor shape: {img_batch.shape}')
print(f'Pixel value range: [{img_array.min():.3f}, {img_array.max():.3f}]')
print(f'Mean pixel value: {img_array.mean():.3f}')
print()
print('This 4D tensor (1, 224, 224, 3) is what gets fed to the ML model!')

## Cell 6: Error Handling Patterns

Robust APIs handle errors gracefully and return structured error responses.

In [ ]:
# Demonstrate different error scenarios and responses

error_scenarios = [
    {
        'scenario': 'Invalid file type (PDF uploaded instead of image)',
        'status_code': 400,
        'response': {
            'success': False,
            'error': {
                'code': 'INVALID_FILE_TYPE',
                'message': 'Expected image/jpeg or image/png, received application/pdf',
                'hint': 'Please upload a JPEG or PNG photo of the plant leaf'
            }
        }
    },
    {
        'scenario': 'Image too large (50 MB photo)',
        'status_code': 413,
        'response': {
            'success': False,
            'error': {
                'code': 'FILE_TOO_LARGE',
                'message': 'File size 52 MB exceeds maximum 10 MB',
                'hint': 'Compress the image before uploading'
            }
        }
    },
    {
        'scenario': 'Model not yet loaded (server starting up)',
        'status_code': 503,
        'response': {
            'success': False,
            'error': {
                'code': 'MODEL_NOT_READY',
                'message': 'ML model is loading. Please retry in a few seconds.',
                'hint': 'Try again after 5 seconds'
            }
        }
    },
    {
        'scenario': 'Server exception during inference',
        'status_code': 500,
        'response': {
            'success': False,
            'error': {
                'code': 'INTERNAL_ERROR',
                'message': 'An unexpected error occurred',
                'hint': 'Please report this to the developer'
            }
        }
    }
]

import json

for scenario in error_scenarios:
    print(f'\nScenario: {scenario["scenario"]}')
    print(f'HTTP Status: {scenario["status_code"]}')
    print(f'Response: {json.dumps(scenario["response"], indent=4)}')
    print('-' * 60)

## Cell 7: Complete backend/main.py Walkthrough

Let's read and understand the actual LeafGuard AI backend file.

In [ ]:
import os

# Read the actual backend main.py file from the repository
repo_root = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath('.'))) if os.path.abspath('.') != '/' else '.', '..')
main_py_path = os.path.join('..', '..', 'backend-api', 'main.py')

try:
    with open(main_py_path, 'r') as f:
        content = f.read()
    print(content)
except FileNotFoundError:
    print('main.py not found at relative path. Navigate to notebooks/week-04/ to run this cell.')
    print()
    print('Expected path: backend-api/main.py')
    print('The file exists in the repository — run this notebook from the notebooks/week-04/ directory.')

## Cell 8: Running the Server and Testing

How to run and test the FastAPI backend.

In [ ]:
# This cell shows you the commands to run — don't execute directly in notebook
# Copy these to your terminal

startup_commands = '''
# 1. Navigate to backend directory
cd backend-api/

# 2. Create and activate virtual environment
python3 -m venv .venv
source .venv/bin/activate        # Linux/Mac
# .venv\\Scripts\\activate        # Windows

# 3. Install dependencies
pip install -r requirements.txt

# 4. Start the server
uvicorn main:app --reload --host 0.0.0.0 --port 8000

# 5. Server starts at:
#    Local:   http://localhost:8000
#    Network: http://YOUR_IP:8000
#    API docs: http://localhost:8000/docs
'''

test_commands = '''
# Test health endpoint
curl http://localhost:8000/health

# Test prediction with curl (replace with a real image)
curl -X POST http://localhost:8000/predict \\
     -F "file=@/path/to/leaf.jpg"

# Using Python requests library:
import requests
with open('leaf.jpg', 'rb') as f:
    response = requests.post(
        'http://localhost:8000/predict',
        files={'file': ('leaf.jpg', f, 'image/jpeg')}
    )
print(response.json())
'''

print('=== STARTUP COMMANDS ===')
print(startup_commands)
print('=== TESTING COMMANDS ===')
print(test_commands)

## Cell 9: Android ↔ FastAPI Contract

The contract between the Android app and the backend must be agreed upon by both sides.

In [ ]:
# Document the exact API contract that Android Retrofit must match

api_contract = {
    'POST /predict': {
        'description': 'Upload a plant leaf image and receive disease prediction',
        'request': {
            'content_type': 'multipart/form-data',
            'fields': {
                'file': 'image/jpeg or image/png, max 10 MB (REQUIRED)'
            }
        },
        'success_response': {
            'status': 200,
            'body': {
                'success': True,
                'disease': 'string (e.g. Tomato Early Blight)',
                'confidence': 'float 0.0-1.0',
                'is_healthy': 'boolean',
                'symptoms': ['string array'],
                'treatment': 'string',
                'prevention': ['string array'],
                'processing_time_ms': 'float'
            }
        },
        'error_responses': {
            400: 'Invalid file type or corrupt image',
            413: 'File exceeds 10 MB',
            422: 'Missing required field',
            503: 'Model not loaded yet'
        }
    },
    'GET /health': {
        'description': 'Check if server and model are ready',
        'success_response': {
            'status': 200,
            'body': {
                'status': '"healthy"',
                'model_loaded': 'boolean'
            }
        }
    }
}

import json
print('LeafGuard AI API Contract:')
print(json.dumps(api_contract, indent=2))
print()
print('Android Retrofit must implement:')
print('''
// ApiService.java
public interface ApiService {
    @Multipart
    @POST("predict")
    Call<PredictionResponse> predict(
        @Part MultipartBody.Part file
    );

    @GET("health")
    Call<HealthResponse> health();
}
''')

## Cell 10: Checkpoint Exercise

Complete these tasks to verify your understanding:

1. ☐ Start the backend server with `uvicorn main:app --reload`
2. ☐ Open `http://localhost:8000/docs` — explore the Swagger UI
3. ☐ Test `/health` from the Swagger UI
4. ☐ Modify the mock response to include plant name in the result
5. ☐ Add a new endpoint `GET /diseases` that returns a list of 5 disease names
6. ☐ Test your new endpoint from Swagger UI

**Bonus challenge:** Add input validation to reject images smaller than 50×50 pixels.

In [ ]:
# Exercise verification: Test your understanding

questions = [
    ('What HTTP method does Android use to upload an image?', 'POST'),
    ('What status code means "I received your request but the file type is wrong"?', '400'),
    ('What is the purpose of the Content-Type: multipart/form-data header?', 'Signals binary data in the request body'),
    ('What Pydantic class would you extend to define a response model?', 'BaseModel'),
    ('What status code does FastAPI automatically return for schema validation failures?', '422'),
]

print('Self-check questions:')
for i, (question, answer) in enumerate(questions, 1):
    print(f'\nQ{i}: {question}')
    input_answer = input('Your answer: ').strip().lower()
    if input_answer:
        correct = answer.lower() in input_answer or input_answer in answer.lower()
        status = '✅ Correct!' if correct else f'❌ Expected: {answer}'
        print(status)
    else:
        print(f'  Answer: {answer}')

## Summary

### What you learned in Week 04:

| Concept | Key takeaway |
|---------|-------------|
| HTTP protocol | Request-response, method + URL + headers + body |
| FastAPI | Modern Python API framework with auto-validation and auto-docs |
| Pydantic models | Type-safe data validation using Python class definitions |
| File uploads | `UploadFile` type handles multipart/form-data |
| Error handling | `HTTPException` for business errors, exception handlers for unexpected errors |
| Status codes | 200 success, 400 bad request, 413 too large, 422 schema error, 500 server error |

### Next week:
- **Week 05:** Connect Android to this API using Retrofit
- **Week 06:** Replace the mock prediction with real ML model inference

---
*See also: `roadmap/week-04-fastapi-backend/learning-notes.md` for comprehensive notes*  
*And: `backend-api/main.py` for the complete working implementation*